# Week 6: Constrained optimality and duality
Complete the `TODO` sections during the 80-minute seminar. Convention throughout: inequality constraints have the form $g_i(x)\leq0$, their multipliers satisfy $\lambda_i\geq0$, and equality multipliers are unrestricted.

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.optimize import minimize
plt.rcParams.update({'figure.figsize': (6, 4), 'axes.grid': True})
TOL = 1e-8

## 1. Construct a Lagrangian
Minimize $(x-2)^2+(y-1)^2$ subject to $x+y=1$, $x\geq0$, and $y\geq0$. Define every constraint in the course convention and print the Lagrangian and its stationarity equations.

In [ ]:
x, y = sp.symbols('x y', real=True)
lam_x, lam_y, nu = sp.symbols('lam_x lam_y nu', real=True)
f_sym = (x-2)**2 + (y-1)**2
g_x, g_y = -x, -y
h = x + y - 1
L_sym = f_sym + lam_x*g_x + lam_y*g_y + nu*h
print('Lagrangian:', sp.expand(L_sym))
print('stationarity:', sp.diff(L_sym, x), '=', 0, ';', sp.diff(L_sym, y), '=', 0)
# TODO: state which multipliers must be nonnegative and which are unrestricted.


## 2. Equality multiplier method
First solve with only $h=0$. Then check the two omitted inequalities and decide whether their multipliers may be zero.

In [ ]:
L_eq = f_sym + nu*h
eq_system = [sp.diff(L_eq, x), sp.diff(L_eq, y), h]
eq_candidates = sp.solve(eq_system, (x, y, nu), dict=True)
print(eq_candidates)
# TODO: substitute the point into g_x and g_y. Record the active set.
# TODO: verify stationarity of the full Lagrangian with lam_x=lam_y=0.


## 3. An inequality and its active set
Minimize $(x-2)^2+(y-1)^2$ subject to $x+y\leq1.5$. Analyze the inactive and active cases separately. Reject candidates using primal feasibility, $\lambda\geq0$, and complementary slackness.

In [ ]:
lam = sp.symbols('lam', real=True)
g_sum = x + y - sp.Rational(3, 2)
L_ineq = f_sym + lam*g_sum
stationarity = [sp.diff(L_ineq, x), sp.diff(L_ineq, y)]
inactive_candidates = sp.solve(stationarity + [lam], (x, y, lam), dict=True)
active_candidates = sp.solve(stationarity + [g_sum], (x, y, lam), dict=True)
print('inactive:', inactive_candidates)
print('active:', active_candidates)
# TODO: evaluate g_sum, lam, and lam*g_sum for each candidate.
# TODO: keep valid candidates and explain why convexity makes the survivor global.


## 4. Active constraint with zero multiplier
Solve $\min_x x^2$ subject to $-x\leq0$. This tests the difference between active and strictly binding in the multiplier sense.

In [ ]:
z, lam0 = sp.symbols('z lam0', real=True)
L_zero = z**2 - lam0*z
zero_candidates = sp.solve([sp.diff(L_zero, z), z], (z, lam0), dict=True)
print(zero_candidates)
# TODO: report the active set and verify all four KKT conditions.


## 5. A reusable KKT audit
The function below uses a list of inequality functions $g_i(x)$ and an optional equality vector $h(x)$. Complete the separate residuals. State a tolerance before declaring a constraint active.

In [ ]:
def kkt_audit(x, grad_f, g, jac_g, lam, h=None, jac_h=None, nu=None, active_tol=1e-7):
    x = np.asarray(x, dtype=float)
    gval = np.atleast_1d(np.asarray(g(x), dtype=float))
    Jg = np.atleast_2d(np.asarray(jac_g(x), dtype=float))
    lam = np.atleast_1d(np.asarray(lam, dtype=float))
    hval = np.zeros(0) if h is None else np.atleast_1d(np.asarray(h(x), dtype=float))
    Jh = np.zeros((0, x.size)) if jac_h is None else np.atleast_2d(np.asarray(jac_h(x), dtype=float))
    nu = np.zeros(0) if nu is None else np.atleast_1d(np.asarray(nu, dtype=float))
    # TODO: compute stationarity = grad_f(x) + Jg.T @ lam + Jh.T @ nu
    # TODO: return stationarity, inequality, equality, dual, and complementarity residuals.
    # TODO: include active_indices = np.flatnonzero(abs(gval) <= active_tol).
    raise NotImplementedError


## 6. Verify with SciPy
SLSQP expects inequality callbacks $c(x)\geq0$. Our constraint $g(x)\leq0$ therefore becomes $c(x)=-g(x)$. Provide analytic Jacobians and compare numerical and symbolic points.

In [ ]:
def f_np(v):
    return float((v[0]-2)**2 + (v[1]-1)**2)

def grad_np(v):
    return np.array([2*(v[0]-2), 2*(v[1]-1)], dtype=float)

eq_constraint = {'type': 'eq', 'fun': lambda v: v[0]+v[1]-1,
                 'jac': lambda v: np.array([1.0, 1.0])}
ineq_constraint = {'type': 'ineq', 'fun': lambda v: 1.5-v[0]-v[1],
                   'jac': lambda v: np.array([-1.0, -1.0])}
res_eq = minimize(f_np, [0.0, 0.0], jac=grad_np, constraints=[eq_constraint],
                  method='SLSQP', options={'ftol': 1e-12})
res_ineq = minimize(f_np, [0.0, 0.0], jac=grad_np, constraints=[ineq_constraint],
                    method='SLSQP', options={'ftol': 1e-12})
print('equality:', res_eq.x, res_eq.fun, res_eq.success)
print('inequality:', res_ineq.x, res_ineq.fun, res_ineq.success)
# TODO: compare with symbolic candidates and reconstruct multipliers from stationarity.
# TODO: pass both solutions through kkt_audit.


## 7. Derive the dual function
Minimize $\tfrac12(x-2)^2$ subject to $x-b\leq0$. Derive $q(\lambda)$ symbolically by minimizing the Lagrangian over the unrestricted variable $x$; then maximize it over $\lambda\geq0$.

In [ ]:
b_sym, lam_sym = sp.symbols('b lam', real=True)
L_scalar = sp.Rational(1, 2)*(x-2)**2 + lam_sym*(x-b_sym)
x_min_L = sp.solve(sp.diff(L_scalar, x), x)[0]
q_sym = sp.simplify(L_scalar.subs(x, x_min_L))
print('argmin_x L =', x_min_L)
print('dual function q =', q_sym)
# TODO: solve max_{lam >= 0} q for b<2 and b>=2.
# TODO: compare p* and d* for b=1.


In [ ]:
b_value = 1.0
lambda_grid = np.linspace(0, 3, 300)
q_numeric = sp.lambdify((lam_sym, b_sym), q_sym, 'numpy')
plt.plot(lambda_grid, q_numeric(lambda_grid, b_value), label='dual lower bound')
# TODO: add a horizontal primal-value line and mark the dual maximizer.
plt.xlabel('lambda'); plt.ylabel('value'); plt.legend(); plt.show()


## 8. Sensitivity of the optimal value
For $b<2$, the multiplier is $2-b$. For $b\geq2$, the constraint is inactive. Compare a central difference of $p(b)$ with $-\lambda^*(b)$ on both sides of the transition.

In [ ]:
def primal_value(b):
    x_star = min(2.0, float(b))
    return 0.5*(x_star-2.0)**2

def optimal_multiplier(b):
    return max(2.0-float(b), 0.0)

def central_difference(fun, b, step=1e-5):
    return (fun(b+step)-fun(b-step))/(2*step)

b_values = np.linspace(0.0, 3.0, 121)
finite_diff = np.array([central_difference(primal_value, b) for b in b_values])
negative_lambda = -np.array([optimal_multiplier(b) for b in b_values])
# TODO: plot both curves, label b=2, and explain the sign.
# TODO: report a numerical error away from the transition.


## 9. Optional extension: two resource limits
Minimize $\tfrac12\|x-a\|^2$ subject to $x_1+x_2\leq b_1$ and $x_1\leq b_2$. Use SLSQP for several bounds, reconstruct multipliers from the active constraints, and compare finite differences of the optimal value. Note where the active set changes.

In [ ]:
a = np.array([2.0, 1.0])
def solve_two_resources(b1, b2):
    objective = lambda v: 0.5*np.dot(v-a, v-a)
    gradient = lambda v: v-a
    constraints = [
        {'type': 'ineq', 'fun': lambda v: b1-v[0]-v[1],
         'jac': lambda v: np.array([-1.0, -1.0])},
        {'type': 'ineq', 'fun': lambda v: b2-v[0],
         'jac': lambda v: np.array([-1.0, 0.0])},
    ]
    return minimize(objective, a.copy(), jac=gradient, constraints=constraints,
                    method='SLSQP', options={'ftol': 1e-12})
# TODO: vary one bound at a time and record the active set, multipliers, and value.


## Final checks
For each solved case, record the point, objective, multipliers, active set, and all KKT residuals. Explain why an active constraint may have zero multiplier and why the derivative with respect to a relaxed upper bound equals the negative multiplier under regularity.